**Python - Data Preparation and Machine Learning** |
This section covers feature preparation, model training, and evaluation to assess how effectively the model distinguishes fraudulent transactions from legitimate ones.

In [0]:
# Access dataset
df = spark.table("workspace.fintech_fraud_db.paysim_raw")
display(df.limit(10))

step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
500,CASH_OUT,37644.46,C1095488083,153620.0,115975.54,C133700066,0.0,37644.46,0,0
500,CASH_OUT,174109.44,C720469896,115975.54,0.0,C1043970373,1471221.71,1645331.15,0,0
500,TRANSFER,91156.75,C2096167907,100945.0,9788.25,C772269814,0.0,91156.75,0,0
500,PAYMENT,3577.83,C1450845331,9788.25,6210.42,M1237105989,0.0,0.0,0,0
500,CASH_OUT,285731.3,C1754302149,9395.0,0.0,C631141908,469326.67,755057.98,0,0
500,CASH_IN,182108.65,C331630551,100114.0,282222.65,C1696461082,311609.97,129501.32,0,0
500,CASH_IN,118899.34,C337487744,2294.0,121193.34,C1270158265,808641.5,689742.16,0,0
500,CASH_OUT,9952.33,C954693331,225070.0,215117.67,C1383116869,8823.19,18775.52,0,0
500,CASH_OUT,158469.5,C1420655012,61297.0,0.0,C596419427,2690503.82,2848973.32,0,0
500,PAYMENT,11541.34,C1701209625,17012.0,5470.66,M473889938,0.0,0.0,0,0


In [0]:
#Inspect data structure
df.printSchema()

root
 |-- step: long (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: long (nullable = true)
 |-- isFlaggedFraud: long (nullable = true)



In [0]:
# Access the existing PaySim table
df = spark.table("workspace.fintech_fraud_db.paysim_raw")

# Select the features and target variable
model_df = df.select(
    "step",
    "type",
    "amount",
    "isFraud"
)

# Display the first 10 rows
display(model_df.limit(10))

step,type,amount,isFraud
500,CASH_OUT,37644.46,0
500,CASH_OUT,174109.44,0
500,TRANSFER,91156.75,0
500,PAYMENT,3577.83,0
500,CASH_OUT,285731.3,0
500,CASH_IN,182108.65,0
500,CASH_IN,118899.34,0
500,CASH_OUT,9952.33,0
500,CASH_OUT,158469.5,0
500,PAYMENT,11541.34,0


In [0]:
#Checking for missing values
from pyspark.sql.functions import col, sum

model_df.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in model_df.columns]
).show()

+----+----+------+-------+
|step|type|amount|isFraud|
+----+----+------+-------+
|   0|   0|     0|      0|
+----+----+------+-------+



In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# Convert transaction categories into numerical indexes
indexer = StringIndexer(
    inputCol="type",
    outputCol="type_index"
)

indexed_df = indexer.fit(model_df).transform(model_df)

# One-hot encode the transaction categories
encoder = OneHotEncoder(
    inputCol="type_index",
    outputCol="type_encoded"
)

encoded_df = encoder.fit(indexed_df).transform(indexed_df)

display(encoded_df.select("type", "type_index", "type_encoded").limit(10))

type,type_index,type_encoded
PAYMENT,1.0,"{""type"":""0"",""size"":""4"",""indices"":[""1""],""values"":[""1.0""]}"
CASH_IN,2.0,"{""type"":""0"",""size"":""4"",""indices"":[""2""],""values"":[""1.0""]}"
TRANSFER,3.0,"{""type"":""0"",""size"":""4"",""indices"":[""3""],""values"":[""1.0""]}"
PAYMENT,1.0,"{""type"":""0"",""size"":""4"",""indices"":[""1""],""values"":[""1.0""]}"
CASH_OUT,0.0,"{""type"":""0"",""size"":""4"",""indices"":[""0""],""values"":[""1.0""]}"
CASH_OUT,0.0,"{""type"":""0"",""size"":""4"",""indices"":[""0""],""values"":[""1.0""]}"
CASH_IN,2.0,"{""type"":""0"",""size"":""4"",""indices"":[""2""],""values"":[""1.0""]}"
PAYMENT,1.0,"{""type"":""0"",""size"":""4"",""indices"":[""1""],""values"":[""1.0""]}"
CASH_OUT,0.0,"{""type"":""0"",""size"":""4"",""indices"":[""0""],""values"":[""1.0""]}"
PAYMENT,1.0,"{""type"":""0"",""size"":""4"",""indices"":[""1""],""values"":[""1.0""]}"


In [0]:
# Create a 10% sample while preserving the class proportions
# Access the original dataset and select the modeling columns
df = spark.table("workspace.fintech_fraud_db.paysim_raw")

model_df = df.select(
    "step",
    "type",
    "amount",
    "isFraud"
)

# Sample the data while preserving class proportions
sampled_df = model_df.sampleBy(
    "isFraud",
    fractions={0: 0.10, 1: 0.10},
    seed=42
)

# Convert the sample to Pandas
pandas_df = sampled_df.toPandas()

print(pandas_df.shape)

(636912, 4)


In [0]:
# Check the distribution of legitimate and fraudulent transactions
print(pandas_df["isFraud"].value_counts())

# Calculate the percentage of each class
print(pandas_df["isFraud"].value_counts(normalize=True) * 100)

isFraud
0    636059
1       853
Name: count, dtype: int64
isFraud
0    99.866073
1     0.133927
Name: proportion, dtype: float64


In [0]:
# Import Scikit-learn preprocessing tools
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Separate input features and target variable
X = pandas_df[["step", "type", "amount"]]
y = pandas_df["isFraud"]

# Convert transaction type into numerical features
preprocessor = ColumnTransformer(
    transformers=[
        ("type", OneHotEncoder(handle_unknown="ignore"), ["type"])
    ],
    remainder="passthrough"
)

# Apply preprocessing
X_processed = preprocessor.fit_transform(X)

print("Feature matrix shape:", X_processed.shape)
print("Target variable shape:", y.shape)

Feature matrix shape: (636912, 7)
Target variable shape: (636912,)


In [0]:
# Access the dataset and select the modeling columns
df = spark.table("workspace.fintech_fraud_db.paysim_raw")

model_df = df.select(
    "step",
    "type",
    "amount",
    "isFraud"
)

# Sample 10% of each classification group
sampled_df = model_df.sampleBy(
    "isFraud",
    fractions={0: 0.10, 1: 0.10},
    seed=42
)

# Convert to Pandas
pandas_df = sampled_df.toPandas()

# Separate input features and target
X = pandas_df[["step", "type", "amount"]]
y = pandas_df["isFraud"]

# Encode the transaction category
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("type", OneHotEncoder(handle_unknown="ignore"), ["type"])
    ],
    remainder="passthrough"
)

X_processed = preprocessor.fit_transform(X)

# Split into training and testing data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_processed,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [0]:
print(X_processed.shape)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(636912, 7)
(509529, 7)
(127383, 7)
(509529,)
(127383,)


In [0]:
from sklearn.linear_model import LogisticRegression

# Initialize the classification model
model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [0]:
# Generate predictions using the test dataset
y_pred = model.predict(X_test)

# Display the first 20 predictions
print(y_pred[:20])

[1 0 1 1 1 0 0 0 0 0 0 0 1 1 0 0 0 0 1 0]


In [0]:
from sklearn.metrics import confusion_matrix

# Compare predicted classifications with actual classifications
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[98506 28706]
 [   18   153]]


In [0]:
#Are legitimate transactions being incorrectly flagged as fraudulent because their amounts are unusually high?
import numpy as np

y_true = y_test.to_numpy()
amount_test = X_test[:, -1]   # amount is the last column after preprocessing

false_alarm = (y_true == 0) & (y_pred == 1)
cleared     = (y_true == 0) & (y_pred == 0)

print("Median amount, false alarms:", np.median(amount_test[false_alarm]))
print("Median amount, correctly cleared:", np.median(amount_test[cleared]))


Median amount, false alarms: 227159.905
Median amount, correctly cleared: 37133.05


In [0]:
type_cols = X_test[:, :5]   # five one-hot type columns come first
types = ["CASH_IN", "CASH_OUT", "DEBIT", "PAYMENT", "TRANSFER"]
for i, t in enumerate(types):
    mask = type_cols[:, i] == 1
    print(t, "false alarms:", (false_alarm & mask).sum(),
          "| median amount:", np.median(amount_test[false_alarm & mask]) if (false_alarm & mask).any() else None)

CASH_IN false alarms: 0 | median amount: None
CASH_OUT false alarms: 18203 | median amount: 173377.09
DEBIT false alarms: 0 | median amount: None
PAYMENT false alarms: 0 | median amount: None
TRANSFER false alarms: 10503 | median amount: 485333.53


In [0]:
for i, t in [(1, "CASH_OUT"), (4, "TRANSFER")]:
    m = type_cols[:, i] == 1
    legit = (y_true == 0) & m
    print(t, "| legitimate in test:", legit.sum(),
          "| flagged:", (false_alarm & m).sum(),
          "| median amount flagged:", np.median(amount_test[false_alarm & m]),
          "| median amount cleared:", np.median(amount_test[cleared & m]) if (cleared & m).any() else None)

CASH_OUT | legitimate in test: 44485 | flagged: 18203 | median amount flagged: 173377.09 | median amount cleared: 129074.66
TRANSFER | legitimate in test: 10503 | flagged: 10503 | median amount flagged: 485333.53 | median amount cleared: None


In [0]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    target_names=["Legitimate (0)", "Fraud (1)"]
))

                precision    recall  f1-score   support

Legitimate (0)       1.00      0.77      0.87    127212
     Fraud (1)       0.01      0.89      0.01       171

      accuracy                           0.77    127383
     macro avg       0.50      0.83      0.44    127383
  weighted avg       1.00      0.77      0.87    127383



In [0]:
# Count transactions flagged as potentially fraudulent
total_flagged = 28706 + 153

# Calculate percentages
false_positive_pct = (28706 / total_flagged) * 100
true_positive_pct = (153 / total_flagged) * 100

print(f"False positives: {false_positive_pct:.2f}%")
print(f"Actually fraudulent: {true_positive_pct:.2f}%")

False positives: 99.47%
Actually fraudulent: 0.53%
